Extraction audit + dictionary-vs-LLM comparison (Day 13).
Part 1: precision audit (excel, r, workday, communication samples)
Part 2: recall audit (gap check on tech postings) -> dictionary iteration
Part 3: ai_query() LLM comparison on ~400-posting sample
        (if ai_query unavailable on Free Edition: documented + deferred)

In [0]:
try:
    result = spark.sql("""
        SELECT ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            'Reply with exactly one word: hello'
        ) AS reply
    """)
    result.display()
    print("✅ ai_query available — Part 3 is on")
except Exception as e:
    print("❌ ai_query not available:")
    print(str(e)[:500])

In [0]:
from pyspark.sql import functions as F

sk = spark.table("jobmarket.silver.silver_posting_skills")
sp = spark.table("jobmarket.silver.silver_job_postings")

def audit_sample(skill_name, pattern, n=20):
    """
    Random matched postings for a skill, snippet around the TRUE match
    position (regexp_instr = the occurrence the boundary matcher saw,
    not the first substring like instr).
    pattern: the regex used for matching, e.g. r'\\bexcel\\b'
             for contains-matched skills pass the plain alias, e.g. 'workday'
    """
    return (sk.filter(F.col("skill") == skill_name)
        .join(sp.select("posting_id", F.lower("description").alias("d")),
              "posting_id")
        .withColumn("pos", F.expr(f"regexp_instr(d, '{pattern}')"))
        .filter("pos > 0")
        .withColumn("snippet",
            F.expr("substring(d, greatest(1, pos - 60), 130)"))
        .select("posting_id", "snippet")
        .orderBy(F.rand(seed=13))
        .limit(n))

In [0]:
audit_sample("excel", r"\\bexcel\\b").display()

In [0]:
audit_sample("r", r"\\br\\b").display()

In [0]:
audit_sample("workday", r"\\bworkday\\b").display()

In [0]:
audit_sample("communication", "communication skills").display()

In [0]:
sp = spark.table("jobmarket.silver.silver_job_postings")

tech_sample = (sp.filter(
        F.col("title_norm").rlike("data engineer|data analyst|data scientist|software engineer|machine learning"))
    .filter(F.col("source") == "kaggle_backfill")   # full descriptions only
    .select("posting_id", "title_norm", "description")
    .orderBy(F.rand(seed=13))
    .limit(10))

# each posting's description alongside what we extracted
gap_view = (tech_sample
    .join(sk.groupBy("posting_id")
            .agg(F.sort_array(F.collect_list("skill")).alias("extracted")),
          "posting_id", "left")
    .select("title_norm", "extracted", "description"))

gap_view.display()

In [0]:
sp = spark.table("jobmarket.silver.silver_job_postings")

llm_sample = (sp.filter(
        F.col("title_norm").rlike(
            "data engineer|data analyst|data scientist|software engineer|machine learning|analytics engineer|business intelligence"))
    .filter(F.col("source") == "kaggle_backfill")
    .filter(F.col("description").isNotNull())
    .select("posting_id", "posted_date",
            F.substring("description", 1, 4000).alias("desc"))
    .orderBy(F.rand(seed=13))
    .limit(400))

prompt_prefix = (
    "Extract technical and professional skills explicitly mentioned in this "
    "job description. Return ONLY a JSON array of lowercase skill names, "
    "nothing else. Example: [\"python\", \"sql\", \"tableau\"]. Description: "
)

llm_raw = llm_sample.withColumn(
    "llm_response",
    F.expr(f"""ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        concat('{prompt_prefix}', desc)
    )"""))

llm_raw.write.format("delta").mode("overwrite") \
    .saveAsTable("jobmarket.silver.llm_extraction_raw")

print("Rows saved:",
      spark.table("jobmarket.silver.llm_extraction_raw").count())

In [0]:
from pyspark.sql.types import ArrayType, StringType

raw = spark.table("jobmarket.silver.llm_extraction_raw")

parsed = (raw
    .withColumn("cleaned",
        F.regexp_replace(F.col("llm_response"), r"```json|```", ""))
    .withColumn("llm_skills",
        F.from_json(F.trim("cleaned"), ArrayType(StringType()))))

n_total  = parsed.count()
n_parsed = parsed.filter(F.col("llm_skills").isNotNull()).count()
print(f"Parsed cleanly: {n_parsed}/{n_total} ({n_parsed/n_total*100:.1f}%)")

parsed.filter(F.col("llm_skills").isNull()) \
      .select(F.substring("llm_response", 1, 200).alias("failed_sample")) \
      .limit(5).display()

In [0]:
# dictionary skills for the same 400 postings
dict_skills = (spark.table("jobmarket.silver.silver_posting_skills")
    .join(parsed.select("posting_id"), "posting_id")
    .groupBy("posting_id")
    .agg(F.collect_set("skill").alias("dict_skills")))

# light normalization of LLM terms -> our canonical names, else keep as-is
synonym_map = {
    "postgres": "postgresql", "ms excel": "excel", "microsoft excel": "excel",
    "power bi": "power bi", "powerbi": "power bi", "amazon web services": "aws",
    "google cloud platform": "gcp", "google cloud": "gcp", "ms sql server": "sql server",
    "reactjs": "react", "react.js": "react", "node": "node.js", "nodejs": "node.js",
    "scikit learn": "scikit-learn", "sklearn": "scikit-learn",
    "apache spark": "spark", "pyspark": "spark", "tensor flow": "tensorflow",
    "k8s": "kubernetes", "ci/cd pipelines": "ci/cd", "cicd": "ci/cd",
    "restful apis": "rest api", "rest apis": "rest api", "restful": "rest api",
    "communication skills": "communication", "problem-solving": "problem solving",
}
map_expr = F.create_map([F.lit(x) for pair in synonym_map.items() for x in pair])

llm_norm = (parsed
    .select("posting_id", F.explode("llm_skills").alias("raw_skill"))
    .withColumn("skill", F.coalesce(map_expr[F.col("raw_skill")], F.col("raw_skill")))
    .groupBy("posting_id")
    .agg(F.collect_set("skill").alias("llm_skills")))

# three-way split per posting
comp = (parsed.select("posting_id")
    .join(dict_skills, "posting_id", "left")
    .join(llm_norm, "posting_id", "left")
    .withColumn("dict_skills", F.coalesce("dict_skills", F.array()))
    .withColumn("llm_skills",  F.coalesce("llm_skills",  F.array()))
    .withColumn("both",      F.array_intersect("dict_skills", "llm_skills"))
    .withColumn("llm_only",  F.array_except("llm_skills", "dict_skills"))
    .withColumn("dict_only", F.array_except("dict_skills", "llm_skills")))

comp.select(
    F.round(F.avg(F.size("both")), 2).alias("avg_agree"),
    F.round(F.avg(F.size("llm_only")), 2).alias("avg_llm_only"),
    F.round(F.avg(F.size("dict_only")), 2).alias("avg_dict_only"),
).display()

# what does each side find that the other misses? (frequency-ranked)
print("--- LLM-only terms (dictionary gaps OR hallucinations/granularity) ---")
comp.select(F.explode("llm_only").alias("s")).groupBy("s").count() \
    .orderBy(F.desc("count")).limit(25).display()

print("--- Dictionary-only (LLM misses OR our false positives) ---")
comp.select(F.explode("dict_only").alias("s")).groupBy("s").count() \
    .orderBy(F.desc("count")).limit(15).display()